# Solar filaments: cross-model consensus and boundary refinement

This notebook generates a new `submission_model.csv` from three competition-trained semantic U-Nets and one
competition-trained YOLO11m segmentation model. It does **not** contain an embedded prediction CSV.

The U-Net encoders are ResNet34, ResNeXt50-32x4d and EfficientNet-B4. Shape/presence maps are averaged, and
YOLO supplies instance proposals. A second YOLO proposal is refined only in a two-pixel boundary band using
the U-Net foreground. A LightGBM regressor learns expected per-annotator matched IoU from geometry,
foreground support and agreement between model families. Greedy selection rejects large overlaps and clips
small overlaps, so all output instances are pixel-disjoint.

**Validation:** YOLO held out network fold 0 (177 images). All fusion training/validation uses only those images.
Meta-validation groups observations by day. The four saved selectors are averaged at test time. Configuration
selection is cross-validated, but the selectors are not retrained inside a second outer loop; the report explicitly
records this limitation. The original network folds grouped by image, not by day.

**Required attached inputs:** competition `filament-segmentation-2026` and the outputs of
`ranjeet258/sf-seg-train-v1`, `sf-seg-train-v2`, `sf-seg-train-v3`, and `sf-yolo-v1`.
They are declared in this notebook's Kaggle metadata. The training source is included below for complete review
and fresh training. Retraining is optional and takes several GPU sessions; default execution uses saved outputs.

**Score provenance:** the older 0.55 result used lamhuy8904's embedded prediction CSV. It is not an independently
reproducible model score and is not generated by this notebook. The new model's public score must be obtained
by actually submitting this notebook's output. `report.json` records local results, not hidden-test claims.

## 1. Dependencies and environment

In [ ]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path
requirements = '''numpy==2.2.6
pandas==2.2.3
opencv-python-headless==4.12.0.88
lightgbm==4.6.0
pycocotools==2.0.10
scipy==1.15.3
scikit-learn==1.6.1
'''
Path('/kaggle/working/requirements.txt').write_text(requirements)
# Pin the numerical post-processing environment for every saved Kaggle run.
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/kaggle/working/requirements.txt'])
print(sys.version)
print(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
os.chdir('/kaggle/working')

## 2. Executable fusion, validation and CSV generation

In [ ]:
%%writefile /kaggle/working/hybrid_v2.py
"""Reproducible U-Net/YOLO instance fusion using only competition-trained models.

Local: python sfseg/exp/hybrid_v2.py
Kaggle: the companion notebook attaches the four original training kernels.
The YOLO model held out network fold 0. All meta-model validation is therefore
restricted to those 177 unseen images, grouped by observation day.
"""
import concurrent.futures
import glob
import hashlib
import json
import os
from pathlib import Path
import pickle
import sys
import time
import zipfile

import cv2
import lightgbm as lgb
import numpy as np
import pandas as pd
from pycocotools import mask as mu
from sklearn.model_selection import GroupKFold

H = W = 2048
ROOT = Path(__file__).resolve().parents[2] if "/kaggle/" not in __file__ else Path("/kaggle/working")
LOCAL = (ROOT / "work/v123/meta.json").exists()
OUT = ROOT / "work/hybrid_v2" if LOCAL else ROOT / "hybrid_v2"
OUT.mkdir(parents=True, exist_ok=True)
T0 = time.time()
FEATURES = ["source", "conf", "area", "bw", "bh", "fill", "aspect", "radius",
            "p0_mean", "p0_q25", "p0_max", "p1_mean", "p1_q25", "p1_max",
            "cc_iou", "yolo_iou", "yolo_conf", "boundary_change", "image_count"]


def log(*args):
    print(f"[{(time.time()-T0)/60:.1f}m]", *args, flush=True)


def encode_crop(mask, x, y):
    """Encode a small bounding-box mask directly into full-image COCO RLE."""
    yy, xx = np.nonzero(mask.T)
    if not len(xx):
        return None
    idx = (yy.astype(np.int64) + x) * H + xx + y
    breaks = np.flatnonzero(np.diff(idx) > 1)
    starts = np.r_[idx[0], idx[breaks + 1]]
    ends = np.r_[idx[breaks] + 1, idx[-1] + 1]
    counts = np.empty(2 * len(starts) + 1, dtype=np.int64)
    counts[0] = starts[0]
    counts[1:-1:2] = ends - starts
    counts[2:-1:2] = starts[1:] - ends[:-1]
    counts[-1] = H * W - ends[-1]
    return mu.frPyObjects({"size": [H, W], "counts": counts.tolist()}, H, W)


def geometry(rle):
    x, y, w, h = map(int, mu.toBbox(rle))
    # Full decode occurs only once per original proposal, not once per setting.
    mask = mu.decode(rle)[y:y+h, x:x+w]
    return x, y, w, h, mask


def counts(gt_sets, pred):
    acc = np.zeros(4, dtype=float)
    for gt in gt_sets:
        if not gt:
            acc[2] += len(pred)
        elif not pred:
            acc[3] += len(gt)
        else:
            iou = np.asarray(mu.iou(pred, gt, [0] * len(gt)))
            hit = iou > .5
            acc += [iou[hit].sum(), hit.sum(), (~hit.any(1)).sum(), (~hit.any(0)).sum()]
    return acc


def pq(c):
    return float(c[0] / max(c[1] + .5 * (c[2] + c[3]), 1))


def discover():
    if LOCAL:
        roots = [ROOT / "work" / v for v in ("v1", "v2", "v3")]
        meta = json.load(open(ROOT / "work/v123/meta.json"))
        gt = pickle.load(open(ROOT / "work/gt.pkl", "rb"))
        yp = ROOT / "work/yolo_v1"
        data = ROOT / "MAGFiLO_1.0_Kaggle_2026"
    else:
        roots = [Path(glob.glob(f"/kaggle/input/**/sf-seg-train-{v}", recursive=True)[0]) for v in ("v1", "v2", "v3")]
        meta = json.load(open(roots[0] / "meta.json"))
        yp = Path(glob.glob("/kaggle/input/**/pred_hold.pkl", recursive=True)[0]).parent
        js = Path(glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0])
        data = js.parent.parent
        coco = json.load(open(js))
        annotations = {}
        for a in coco["annotations"]:
            annotations.setdefault(a["image_id"], []).append(mu.merge(mu.frPyObjects(a["segmentation"], H, W)))
        gt = {}
        for im in coco["images"]:
            gt.setdefault(Path(im["file_name"]).stem, []).append(annotations.get(im["id"], []))
    hold = pickle.load(open(yp / "pred_hold.pkl", "rb"))
    test = pickle.load(open(yp / "pred_test.pkl", "rb"))
    fold = dict(zip(meta["stems"], meta["fold"]))
    assert set(hold) == {s for s, f in fold.items() if f == 0}, "YOLO holdout/fold mismatch"
    return roots, meta, gt, hold, test, data


def one_image(job):
    stem, items, paths, gt_sets = job
    p = np.zeros((H, W, 2), np.float32)
    for path in paths:
        with zipfile.ZipFile(path) as z:
            a = cv2.imdecode(np.frombuffer(z.read(stem + ".png"), np.uint8), cv2.IMREAD_COLOR)
        p += a[:, :, :2]
    p /= 255 * len(paths)
    n, lab, stats, _ = cv2.connectedComponentsWithStats((p[:, :, 0] >= .5).astype(np.uint8), 8)
    proposals = []
    for k in range(1, n):
        x, y, w, h, area = map(int, stats[k])
        if area >= 60:
            proposals.append((0, 0., encode_crop(lab[y:y+h, x:x+w] == k, x, y), 0.))
    cc = [r[2] for r in proposals]
    ys = [{"size": [H, W], "counts": c.encode() if isinstance(c, str) else c}
          for score, c in items if score >= .03]
    scores = [score for score, _ in items if score >= .03]
    for score, rle in zip(scores, ys):
        if mu.area(rle) < 60:
            continue
        proposals.append((1, score, rle, 0.))
        x, y, w, h, mask = geometry(rle)
        x0, y0, x1, y1 = max(0, x-3), max(0, y-3), min(W, x+w+3), min(H, y+h+3)
        m = np.zeros((y1-y0, x1-x0), np.uint8)
        m[y-y0:y-y0+h, x-x0:x-x0+w] = mask
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        refined = (cv2.erode(m, kernel) > 0) | ((cv2.dilate(m, kernel) > 0) & (p[y0:y1, x0:x1, 0] >= .5))
        area = refined.sum()
        if area >= 60 and .7 < area / max(m.sum(), 1) < 1.4:
            change = float(np.logical_xor(m, refined).sum() / max(m.sum(), 1))
            proposals.append((2, score, encode_crop(refined, x0, y0), change))
    if not proposals:
        return []
    rles = [r[2] for r in proposals]
    ci = mu.iou(rles, cc, [0] * len(cc)) if cc else np.zeros((len(rles), 0))
    yi = mu.iou(rles, ys, [0] * len(ys)) if ys else np.zeros((len(rles), 0))
    values = np.zeros(len(rles))
    if gt_sets is not None:
        for gs in gt_sets:
            if gs:
                iou = np.asarray(mu.iou(rles, gs, [0] * len(gs)))
                values += np.where(iou > .5, iou, 0).sum(1)
    rows = []
    for i, (source, score, rle, change) in enumerate(proposals):
        x, y, w, h, m = geometry(rle)
        fg = m > 0
        v = p[y:y+h, x:x+w][fg]
        area = int(fg.sum())
        row = dict(stem=stem, source=source, conf=score, rle=rle, area=area,
                   bw=w, bh=h, fill=area / (w*h), aspect=max(w,h)/max(1,min(w,h)),
                   radius=float(np.hypot(x+w/2-1024,y+h/2-1024)/1000),
                   p0_mean=float(v[:,0].mean()), p0_q25=float(np.quantile(v[:,0],.25)), p0_max=float(v[:,0].max()),
                   p1_mean=float(v[:,1].mean()), p1_q25=float(np.quantile(v[:,1],.25)), p1_max=float(v[:,1].max()),
                   cc_iou=float(ci[i].max()) if len(cc) else 0.,
                   yolo_iou=float(yi[i].max()) if len(ys) else 0.,
                   yolo_conf=float(max((s for s,j in zip(scores,yi[i]) if j >= .5),default=0)),
                   boundary_change=change, image_count=len(cc), v=values[i], A=len(gt_sets) if gt_sets is not None else 0)
        rows.append(row)
    return rows


def build(roots, items, gt, split):
    cache = OUT / f"candidates_{split}.pkl"
    if cache.exists():
        return pd.read_pickle(cache)
    mapping = {}
    for root in roots:
        for path in root.glob(f"{'oof' if split == 'hold' else 'test'}_fold*.zip"):
            with zipfile.ZipFile(path) as z:
                for name in z.namelist():
                    mapping.setdefault(name[:-4], []).append(str(path))
    jobs = [(s, sorted(v, reverse=True), mapping[s], gt[s] if split == "hold" else None) for s,v in sorted(items.items())]
    rows = []
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as pool:
        for i, result in enumerate(pool.map(one_image, jobs)):
            rows.extend(result)
            if i % 20 == 0:
                log(split, i, "of", len(jobs), "candidates", len(rows))
    df = pd.DataFrame(rows)
    df.to_pickle(cache)
    return df


def select(df, cut, sources):
    """Greedy RLE selection, strict disjointness, no full-frame mask copies."""
    result = {}
    for stem, group in df[(df.score >= cut) & df.source.isin(sources)].groupby("stem", sort=False):
        chosen = []
        for row in group.sort_values("score", ascending=False, kind="stable").itertuples():
            r = row.rle
            if chosen:
                occupied = mu.merge(chosen)
                overlap = float(mu.area(mu.merge([r, occupied], intersect=True)))
                if overlap > .1 * row.area:
                    continue
                if overlap:
                    mask = mu.decode(r).astype(bool) & ~mu.decode(occupied).astype(bool)
                    if mask.sum() < 60:
                        continue
                    r = mu.encode(np.asfortranarray(mask.astype(np.uint8)))
            chosen.append(r)
        result[stem] = chosen
    return result


def evaluate(gt, pred, stems):
    total = sum((counts(gt[s], pred.get(s, [])) for s in stems), np.zeros(4))
    return dict(PQ=pq(total), iou_sum=total[0], TP=int(total[1]), FP=int(total[2]), FN=int(total[3]))


def write_csv(pred, path):
    rows = [(f"{s}_{i}", r["counts"].decode("ascii")) for s, rs in sorted(pred.items()) for i,r in enumerate(rs,1)]
    pd.DataFrame(rows, columns=["filament_id", "segmentation_rle"]).to_csv(path, index=False)
    log("wrote", path, "instances", len(rows))


def main():
    cv2.setNumThreads(1)
    roots, meta, gt, hold, test_items, data = discover()
    oof = build(roots, hold, gt, "hold")
    test = build(roots, test_items, gt, "test")
    stems = sorted(hold)
    # All candidates of an observation/day stay in the same meta-validation fold.
    groups = oof.stem.str[:8]
    oof["score"] = 0.
    oof["meta_fold"] = -1
    test["score"] = 0.
    params = dict(objective="regression", learning_rate=.035, num_leaves=15,
                  min_data_in_leaf=60, lambda_l2=10., feature_fraction=.85,
                  verbosity=-1, num_threads=3, seed=20261028)
    for f, (tr, va) in enumerate(GroupKFold(4).split(oof, groups=groups)):
        model = lgb.train(params, lgb.Dataset(oof.iloc[tr][FEATURES],
                          oof.iloc[tr].v / oof.iloc[tr].A, weight=oof.iloc[tr].A), 250)
        oof.loc[va,"score"] = model.predict(oof.iloc[va][FEATURES])
        oof.loc[va,"meta_fold"] = f
        test["score"] += model.predict(test[FEATURES]) / 4
        model.save_model(str(OUT / f"selector_fold{f}.txt"))
    oof.to_pickle(OUT / "scored_hold.pkl")
    test.to_pickle(OUT / "scored_test.pkl")
    settings = []
    for name, sources in [("semantic",[0]),("raw_hybrid",[0,1]),("refined_hybrid",[0,2]),("all",[0,1,2])]:
        for cut in (.18,.22,.26,.30,.35,.40):
            pred = select(oof, cut, sources)
            exact = evaluate(gt,pred,stems)
            fold_counts = []
            for f in range(4):
                ss = oof.loc[oof.meta_fold == f,"stem"].unique()
                fold_counts.append(sum((counts(gt[s],pred.get(s,[])) for s in ss), np.zeros(4)).tolist())
            settings.append(dict(name=name,sources=sources,cut=cut,**exact,fold_counts=fold_counts))
            log(name,cut,round(exact["PQ"],5))
    # Choose settings using the other three meta-folds, then evaluate held-out fold.
    nested_counts = np.zeros(4)
    for f in range(4):
        choice = max(settings,key=lambda r:pq(np.sum([c for j,c in enumerate(r["fold_counts"]) if j != f],axis=0)))
        nested_counts += choice["fold_counts"][f]
    best = max(settings,key=lambda r:r["PQ"])
    chosen = select(test,best["cut"],best["sources"])
    write_csv(chosen,OUT / "submission_model.csv")
    # Controlled high-precision supplement to the existing reference: preserve all
    # reference masks; add only new masks supported by both model families.
    ref_file = ROOT / "work/champ055.csv"
    fusion = None
    if ref_file.exists():
        ref = {}
        for row in pd.read_csv(ref_file).itertuples():
            ref.setdefault(row.filament_id.rsplit("_",1)[0],[]).append({"size":[H,W],"counts":row.segmentation_rle.encode()})
        supported = test[(test.cc_iou >= .5) & (test.yolo_iou >= .5) & (test.yolo_conf >= .15)]
        additions = select(supported,max(.40,best["cut"]),best["sources"])
        n_add = 0
        for stem, rs in additions.items():
            base = ref.setdefault(stem,[])
            for r in rs:
                overlap = float(mu.area(mu.merge([r,mu.merge(base)],intersect=True))) if base else 0.
                if overlap > .05 * float(mu.area(r)):
                    continue
                if overlap:
                    m = mu.decode(r).astype(bool) & ~mu.decode(mu.merge(base)).astype(bool)
                    if m.sum() < 100:
                        continue
                    r = mu.encode(np.asfortranarray(m.astype(np.uint8)))
                base.append(r)
                n_add += 1
        write_csv(ref,OUT / "submission_reference_consensus.csv")
        fusion = dict(added=n_add,provenance="lamhuy8904 embedded prediction CSV + independently trained cross-model consensus; reference masks are not reproducible from available weights")
    # Fast RLE validation: areas, bounds, and exact pairwise intersections.
    audits = {}
    for path in OUT.glob("submission*.csv"):
        frame = pd.read_csv(path)
        assert frame.columns.tolist() == ["filament_id","segmentation_rle"]
        assert frame.filament_id.is_unique and frame.notna().all().all()
        overlap = 0
        for stem,g in frame.groupby(frame.filament_id.str.rsplit("_",n=1).str[0]):
            assert stem in test_items
            rles = [{"size":[H,W],"counts":c.encode()} for c in g.segmentation_rle]
            areas = mu.area(rles)
            assert (areas > 0).all()
            overlap += int(areas.sum()) - int(mu.area(mu.merge(rles)))
        assert overlap == 0, (path,overlap)
        audits[path.name] = dict(rows=len(frame),overlap_pixels=overlap,sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    report = dict(chosen=best,nested_meta_PQ=pq(nested_counts),holdout_images=len(stems),
                  network_holdout_fold=0,meta_validation="4 folds grouped by observation day",
                  selection_validation_limit="Network training is independent of these images. Meta-model cross-fitting and settings selection share meta-training data; this is not a fully nested retraining estimate.",
                  settings=settings,fusion=fusion,audits=audits,
                  sources=["ranjeet258/sf-seg-train-v1","ranjeet258/sf-seg-train-v2","ranjeet258/sf-seg-train-v3","ranjeet258/sf-yolo-v1"])
    (OUT / "report.json").write_text(json.dumps(report,indent=2),encoding="utf-8")
    log("chosen",best["name"],best["cut"],"holdout PQ",best["PQ"],"nested settings estimate",pq(nested_counts))
    log("audits",audits)


if __name__ == "__main__":
    main()


In [ ]:
subprocess.run([sys.executable, '-u', '/kaggle/working/hybrid_v2.py'], check=True)

## 3. Results and submission audit

In [ ]:
import pandas as pd
report = json.loads(Path('/kaggle/working/hybrid_v2/report.json').read_text())
print('Chosen configuration:', report['chosen']['name'], 'cut:', report['chosen']['cut'])
print('Local held-out PQ:', report['chosen']['PQ'])
print('Cross-validated settings estimate:', report['nested_meta_PQ'])
print('Validation limitation:', report['selection_validation_limit'])
display(pd.DataFrame(report['settings']).drop(columns=['fold_counts']).sort_values('PQ', ascending=False))
print('CSV checks:', report['audits'])
import shutil
shutil.copyfile('/kaggle/working/hybrid_v2/submission_model.csv', '/kaggle/working/submission.csv')
assert Path('/kaggle/working/submission.csv').stat().st_size > 100
print('Submit /kaggle/working/submission.csv to obtain the public score.')

## 4. Complete original training source

These cells write source files for auditing and retraining. They do not start training during the default run.
The U-Net scripts train four image-grouped folds, soft multi-annotator shape/presence targets, ImageNet initialization,
EMA and eight-view test-time augmentation. YOLO trains on one sample per annotator set with network fold 0 held out.
Use a separate GPU session for each U-Net script. The original scripts write to `/kaggle/working` and use
`/kaggle/temp/cache`; execute one training script per session and preserve its outputs.
Checkpoints and maps are attached through the original kernel outputs. Fresh training additionally needs
`torch`, `torchvision`, `segmentation-models-pytorch` and `ultralytics`; use the training requirements in the source release.
The source fixes seeds, but GPU/library differences can prevent bitwise-identical retraining.

In [ ]:
Path('/kaggle/working/training_source').mkdir(exist_ok=True)

In [ ]:
%%writefile /kaggle/working/training_source/train_v1.py
"""Solar filament segmentation - grouped K-fold training of a 2-channel soft-label U-Net.

Targets (both in [0, 1], built from the per-annotator polygons):
  ch0 "shape":    (#annotators covering the pixel) / (#annotators who drew anything in this filament cluster)
  ch1 "presence": (#annotators covering the pixel) / (#annotators of the image)
A filament cluster is an 8-connected component of the union of all annotators' masks.

Outputs (in /kaggle/working): oof_fold{k}.zip, test_fold{k}.zip (PNG probability maps, uint8, B=ch0 G=ch1),
model_fold{k}.pt, meta.json. Folds are trained in parallel, one process per GPU.
"""
import glob
import io
import json
import math
import multiprocessing as mp
import os
import subprocess
import sys
import time
import zipfile

import numpy as np

CFG = dict(folds=4, crop=768, bs=8, epochs=36, crops_per_img=3, lr=3e-4, wd=1e-4, enc="resnet34",
           seed=42, ema=0.998, loader_workers=2)
CACHE = "/kaggle/temp/cache"
OUT = "/kaggle/working"
H = W = 2048
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f}m]", *a, flush=True)


# ----------------------------------------------------------------------------- data cache (CPU, parent process)
def prepare():
    import cv2
    from pycocotools import mask as mu

    os.makedirs(CACHE, exist_ok=True)
    js = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
    root = os.path.dirname(os.path.dirname(js))
    train_dir = os.path.join(root, "train", "train_images")
    test_dir = os.path.join(root, "test", "test_images")
    coco = json.load(open(js))
    log("labels", js)

    sets_of, anns_of = {}, {}
    for im in coco["images"]:
        sets_of.setdefault(os.path.splitext(im["file_name"])[0], []).append(im["id"])
    for a in coco["annotations"]:
        anns_of.setdefault(a["image_id"], []).append(a)
    stems = sorted(sets_of)
    test_stems = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(test_dir, "*.jpeg")))
    n = len(stems)
    log(f"{n} train images, {len(coco['images'])} annotator-sets, {len(test_stems)} test images")

    imgs = np.lib.format.open_memmap(f"{CACHE}/imgs.npy", "w+", np.uint8, (n, H, W))
    tgt = np.lib.format.open_memmap(f"{CACHE}/tgt.npy", "w+", np.uint8, (n, 2, H, W))
    centers = []
    for i, s in enumerate(stems):
        im = cv2.imread(os.path.join(train_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), (s, None if im is None else im.shape)
        imgs[i] = im
        per_annot, cs = [], []
        for aset in sets_of[s]:
            m = np.zeros((H, W), np.uint8)
            for a in anns_of.get(aset, []):
                m |= mu.decode(mu.merge(mu.frPyObjects(a["segmentation"], H, W)))
                x, y, w, h = a["bbox"]
                cs.append([x + w / 2, y + h / 2])
            per_annot.append(m)
        count = np.sum(per_annot, axis=0).astype(np.float32)
        ncc, lab = cv2.connectedComponents((count > 0).astype(np.uint8), connectivity=8)
        k = np.zeros(ncc, np.float32)
        for m in per_annot:
            k[np.unique(lab[m > 0])] += 1
        k[0] = 1
        tgt[i, 0] = np.rint(255 * count / np.maximum(k[lab], 1)).astype(np.uint8)
        tgt[i, 1] = np.rint(255 * count / len(per_annot)).astype(np.uint8)
        centers.append(cs)
        if i % 100 == 0:
            log("cache train", i)
    imgs.flush(); tgt.flush(); del imgs, tgt

    test = np.lib.format.open_memmap(f"{CACHE}/test.npy", "w+", np.uint8, (len(test_stems), H, W))
    for i, s in enumerate(test_stems):
        im = cv2.imread(os.path.join(test_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), s
        test[i] = im
    test.flush(); del test

    perm = np.random.RandomState(CFG["seed"]).permutation(n)
    fold = np.zeros(n, int)
    fold[perm] = np.arange(n) % CFG["folds"]
    meta = dict(stems=stems, test_stems=test_stems, fold=fold.tolist(), centers=centers,
                n_sets=[len(sets_of[s]) for s in stems], cfg=CFG)
    json.dump(meta, open(f"{CACHE}/meta.json", "w"))
    json.dump({k: v for k, v in meta.items() if k != "centers"}, open(f"{OUT}/meta.json", "w"))
    log("cache done")


# ----------------------------------------------------------------------------- worker (one per GPU)
def worker(gpu, folds):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    import copy
    import cv2
    import torch
    import torch.nn.functional as F
    import segmentation_models_pytorch as smp

    cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark = True
    meta = json.load(open(f"{CACHE}/meta.json"))
    fold_of = np.array(meta["fold"])
    C = CFG["crop"]

    class DS(torch.utils.data.Dataset):
        def __init__(self, idx):
            self.idx, self.imgs, self.tgt = idx, None, None

        def __len__(self):
            return len(self.idx) * CFG["crops_per_img"]

        def __getitem__(self, i):
            if self.imgs is None:
                self.imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
                self.tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
            j = int(self.idx[i % len(self.idx)])
            r = np.random
            win = int(round(C / math.exp(r.uniform(math.log(0.7), math.log(1.4)))))
            cs = meta["centers"][j]
            if cs and r.rand() < 0.6:
                cx, cy = cs[r.randint(len(cs))]
                cx += r.uniform(-win / 3, win / 3); cy += r.uniform(-win / 3, win / 3)
            else:
                cx, cy = r.uniform(0, W), r.uniform(0, H)
            x0 = int(np.clip(cx - win / 2, 0, W - win)); y0 = int(np.clip(cy - win / 2, 0, H - win))
            im = np.array(self.imgs[j, y0:y0 + win, x0:x0 + win])
            tg = np.array(self.tgt[j, :, y0:y0 + win, x0:x0 + win])
            if win != C:
                interp = cv2.INTER_AREA if win > C else cv2.INTER_LINEAR
                im = cv2.resize(im, (C, C), interpolation=interp)
                tg = np.stack([cv2.resize(t, (C, C), interpolation=cv2.INTER_LINEAR) for t in tg])
            k = r.randint(4)
            im, tg = np.rot90(im, k), np.rot90(tg, k, axes=(1, 2))
            if r.rand() < 0.5:
                im, tg = im[:, ::-1], tg[:, :, ::-1]
            x = im.astype(np.float32) / 255.0
            x = np.clip((x - 0.5) * r.uniform(0.8, 1.25) + 0.5 + r.uniform(-0.08, 0.08), 0, 1) ** r.uniform(0.8, 1.25)
            return (torch.from_numpy(np.ascontiguousarray(x))[None],
                    torch.from_numpy(np.ascontiguousarray(tg)).float() / 255.0)

    def norm(x):
        return (x - 0.45) / 0.225

    @torch.no_grad()
    def predict(model, im):
        """uint8 (H, W) -> uint8 (2, H, W) probabilities, 4-flip TTA."""
        x = norm(torch.from_numpy(np.ascontiguousarray(im)).cuda().float()[None, None] / 255.0)
        acc = 0
        for fl in ([], [2], [3], [2, 3]):
            xx = torch.flip(x, fl) if fl else x
            with torch.autocast("cuda", dtype=torch.float16):
                p = torch.sigmoid(model(xx.contiguous(memory_format=torch.channels_last)).float())
            acc = acc + (torch.flip(p, fl) if fl else p)
        return (acc[0] / 4 * 255).round().byte().cpu().numpy()

    def dump(zpath, names, arrays_iter):
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
            for name, p in zip(names, arrays_iter):
                ok, buf = cv2.imencode(".png", np.dstack([p[0], p[1], np.zeros_like(p[0])]))
                assert ok
                z.writestr(name + ".png", buf.tobytes())

    def run_fold(f, smoke=False):
        tag = f"smoke{gpu}" if smoke else f"fold{f}"
        tr = np.flatnonzero(fold_of != f); va = np.flatnonzero(fold_of == f)
        torch.manual_seed(CFG["seed"] + f)
        model = smp.Unet(CFG["enc"], encoder_weights="imagenet", in_channels=1, classes=2)
        model = model.cuda().to(memory_format=torch.channels_last)
        ema = copy.deepcopy(model).eval()
        for p in ema.parameters():
            p.requires_grad_(False)
        dl = torch.utils.data.DataLoader(
            DS(tr), batch_size=CFG["bs"], shuffle=True, drop_last=True, num_workers=CFG["loader_workers"],
            pin_memory=True, persistent_workers=True,
            worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2 ** 31))
        total = 3 if smoke else CFG["epochs"] * len(dl)
        warm = min(300, total // 10 + 1)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
        scaler = torch.amp.GradScaler("cuda")
        step, run, t0 = 0, 0.0, time.time()
        model.train()
        while step < total:
            for x, y in dl:
                x = norm(x.cuda(non_blocking=True)).contiguous(memory_format=torch.channels_last)
                y = y.cuda(non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    logit = model(x).float()
                bce = F.binary_cross_entropy_with_logits(logit, y)
                p = torch.sigmoid(logit)
                dice = 1 - ((2 * (p * y).sum((0, 2, 3)) + 1) / (p.sum((0, 2, 3)) + y.sum((0, 2, 3)) + 1)).mean()
                loss = bce + 0.5 * dice
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); sched.step()
                d = min(CFG["ema"], (1 + step) / (10 + step))
                with torch.no_grad():
                    for pe, pm in zip(ema.parameters(), model.parameters()):
                        pe.lerp_(pm, 1 - d)
                    for be, bm in zip(ema.buffers(), model.buffers()):
                        be.copy_(bm)
                step += 1
                run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
                if step % 200 == 0:
                    log(f"{tag} step {step}/{total} loss {run:.4f} bce {bce.item():.4f} "
                        f"{(time.time() - t0) / step:.2f}s/it")
                if step >= total:
                    break
        del dl

        imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
        tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
        test = np.load(f"{CACHE}/test.npy", mmap_mode="r")
        va_use = va[:2] if smoke else va
        te_use = range(2 if smoke else len(test))
        inter = np.zeros(2); den = np.zeros(2)

        def oof_iter():
            for j in va_use:
                p = predict(ema, imgs[j])
                t = np.asarray(tgt[j]) >= 128
                q = p >= 128
                inter[:] += (q & t).sum((1, 2)); den[:] += q.sum((1, 2)) + t.sum((1, 2))
                yield p

        dump(f"{OUT}/oof_{tag}.zip", [meta["stems"][j] for j in va_use], oof_iter())
        log(f"{tag} OOF pixel dice @0.5: shape {2 * inter[0] / max(den[0], 1):.4f} "
            f"presence {2 * inter[1] / max(den[1], 1):.4f}")
        dump(f"{OUT}/test_{tag}.zip", [meta["test_stems"][j] for j in te_use], (predict(ema, test[j]) for j in te_use))
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in ema.state_dict().items()},
                   f"{OUT}/model_{tag}.pt")
        if smoke:
            for fn in (f"oof_{tag}.zip", f"test_{tag}.zip", f"model_{tag}.pt"):
                os.remove(f"{OUT}/{fn}")
        log(f"{tag} done")
        del model, ema, opt
        torch.cuda.empty_cache()

    log(f"worker gpu {gpu} folds {folds}: {torch.cuda.get_device_name(0)}")
    run_fold(folds[0], smoke=True)      # end-to-end check of train / predict / save before spending GPU time
    for f in folds:
        run_fold(f)


def main():
    # The parent imports neither torch nor cv2, so the forked children start from a clean process.
    import importlib.util
    for mod, pkg in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("pycocotools", "pycocotools")):
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    assert n_gpu >= 1, "no GPU"
    log("gpus", n_gpu, CFG)
    ctx = mp.get_context("fork")
    procs = [ctx.Process(target=prepare)]
    procs[0].start(); procs[0].join()
    assert procs[0].exitcode == 0, "prepare failed"
    procs = [ctx.Process(target=worker, args=(g, list(range(g, CFG["folds"], n_gpu)))) for g in range(n_gpu)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    assert all(p.exitcode == 0 for p in procs), [p.exitcode for p in procs]
    log("all done", sorted(os.listdir(OUT)))


main()


In [ ]:
%%writefile /kaggle/working/training_source/train_v2.py
"""Solar filament segmentation - grouped K-fold training of a 2-channel soft-label U-Net.

Targets (both in [0, 1], built from the per-annotator polygons):
  ch0 "shape":    (#annotators covering the pixel) / (#annotators who drew anything in this filament cluster)
  ch1 "presence": (#annotators covering the pixel) / (#annotators of the image)
A filament cluster is an 8-connected component of the union of all annotators' masks.

Outputs (in /kaggle/working): oof_fold{k}.zip, test_fold{k}.zip (PNG probability maps, uint8, B=ch0 G=ch1),
model_fold{k}.pt, meta.json. Folds are trained in parallel, one process per GPU.
"""
import glob
import io
import json
import math
import multiprocessing as mp
import os
import subprocess
import sys
import time
import zipfile

import numpy as np

CFG = dict(folds=4, crop=768, bs=8, epochs=36, crops_per_img=3, lr=3e-4, wd=1e-4, enc="resnext50_32x4d",
           seed=42, ema=0.998, loader_workers=2)
CACHE = "/kaggle/temp/cache"
OUT = "/kaggle/working"
H = W = 2048
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f}m]", *a, flush=True)


# ----------------------------------------------------------------------------- data cache (CPU, parent process)
def prepare():
    import cv2
    from pycocotools import mask as mu

    os.makedirs(CACHE, exist_ok=True)
    js = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
    root = os.path.dirname(os.path.dirname(js))
    train_dir = os.path.join(root, "train", "train_images")
    test_dir = os.path.join(root, "test", "test_images")
    coco = json.load(open(js))
    log("labels", js)

    sets_of, anns_of = {}, {}
    for im in coco["images"]:
        sets_of.setdefault(os.path.splitext(im["file_name"])[0], []).append(im["id"])
    for a in coco["annotations"]:
        anns_of.setdefault(a["image_id"], []).append(a)
    stems = sorted(sets_of)
    test_stems = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(test_dir, "*.jpeg")))
    n = len(stems)
    log(f"{n} train images, {len(coco['images'])} annotator-sets, {len(test_stems)} test images")

    imgs = np.lib.format.open_memmap(f"{CACHE}/imgs.npy", "w+", np.uint8, (n, H, W))
    tgt = np.lib.format.open_memmap(f"{CACHE}/tgt.npy", "w+", np.uint8, (n, 2, H, W))
    centers = []
    for i, s in enumerate(stems):
        im = cv2.imread(os.path.join(train_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), (s, None if im is None else im.shape)
        imgs[i] = im
        per_annot, cs = [], []
        for aset in sets_of[s]:
            m = np.zeros((H, W), np.uint8)
            for a in anns_of.get(aset, []):
                m |= mu.decode(mu.merge(mu.frPyObjects(a["segmentation"], H, W)))
                x, y, w, h = a["bbox"]
                cs.append([x + w / 2, y + h / 2])
            per_annot.append(m)
        count = np.sum(per_annot, axis=0).astype(np.float32)
        ncc, lab = cv2.connectedComponents((count > 0).astype(np.uint8), connectivity=8)
        k = np.zeros(ncc, np.float32)
        for m in per_annot:
            k[np.unique(lab[m > 0])] += 1
        k[0] = 1
        tgt[i, 0] = np.rint(255 * count / np.maximum(k[lab], 1)).astype(np.uint8)
        tgt[i, 1] = np.rint(255 * count / len(per_annot)).astype(np.uint8)
        centers.append(cs)
        if i % 100 == 0:
            log("cache train", i)
    imgs.flush(); tgt.flush(); del imgs, tgt

    test = np.lib.format.open_memmap(f"{CACHE}/test.npy", "w+", np.uint8, (len(test_stems), H, W))
    for i, s in enumerate(test_stems):
        im = cv2.imread(os.path.join(test_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), s
        test[i] = im
    test.flush(); del test

    perm = np.random.RandomState(CFG["seed"]).permutation(n)
    fold = np.zeros(n, int)
    fold[perm] = np.arange(n) % CFG["folds"]
    meta = dict(stems=stems, test_stems=test_stems, fold=fold.tolist(), centers=centers,
                n_sets=[len(sets_of[s]) for s in stems], cfg=CFG)
    json.dump(meta, open(f"{CACHE}/meta.json", "w"))
    json.dump({k: v for k, v in meta.items() if k != "centers"}, open(f"{OUT}/meta.json", "w"))
    log("cache done")


# ----------------------------------------------------------------------------- worker (one per GPU)
def worker(gpu, folds):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    import copy
    import cv2
    import torch
    import torch.nn.functional as F
    import segmentation_models_pytorch as smp

    cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark = True
    meta = json.load(open(f"{CACHE}/meta.json"))
    fold_of = np.array(meta["fold"])
    C = CFG["crop"]

    class DS(torch.utils.data.Dataset):
        def __init__(self, idx):
            self.idx, self.imgs, self.tgt = idx, None, None

        def __len__(self):
            return len(self.idx) * CFG["crops_per_img"]

        def __getitem__(self, i):
            if self.imgs is None:
                self.imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
                self.tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
            j = int(self.idx[i % len(self.idx)])
            r = np.random
            win = int(round(C / math.exp(r.uniform(math.log(0.7), math.log(1.4)))))
            cs = meta["centers"][j]
            if cs and r.rand() < 0.6:
                cx, cy = cs[r.randint(len(cs))]
                cx += r.uniform(-win / 3, win / 3); cy += r.uniform(-win / 3, win / 3)
            else:
                cx, cy = r.uniform(0, W), r.uniform(0, H)
            x0 = int(np.clip(cx - win / 2, 0, W - win)); y0 = int(np.clip(cy - win / 2, 0, H - win))
            im = np.array(self.imgs[j, y0:y0 + win, x0:x0 + win])
            tg = np.array(self.tgt[j, :, y0:y0 + win, x0:x0 + win])
            if win != C:
                interp = cv2.INTER_AREA if win > C else cv2.INTER_LINEAR
                im = cv2.resize(im, (C, C), interpolation=interp)
                tg = np.stack([cv2.resize(t, (C, C), interpolation=cv2.INTER_LINEAR) for t in tg])
            k = r.randint(4)
            im, tg = np.rot90(im, k), np.rot90(tg, k, axes=(1, 2))
            if r.rand() < 0.5:
                im, tg = im[:, ::-1], tg[:, :, ::-1]
            x = im.astype(np.float32) / 255.0
            x = np.clip((x - 0.5) * r.uniform(0.8, 1.25) + 0.5 + r.uniform(-0.08, 0.08), 0, 1) ** r.uniform(0.8, 1.25)
            return (torch.from_numpy(np.ascontiguousarray(x))[None],
                    torch.from_numpy(np.ascontiguousarray(tg)).float() / 255.0)

    def norm(x):
        return (x - 0.45) / 0.225

    @torch.no_grad()
    def predict(model, im):
        """uint8 (H, W) -> uint8 (2, H, W) probabilities, 8-view dihedral TTA."""
        x0 = norm(torch.from_numpy(np.ascontiguousarray(im)).cuda().float()[None, None] / 255.0)
        acc = 0
        for tr in (False, True):
            x = x0.transpose(2, 3) if tr else x0
            for fl in ([], [2], [3], [2, 3]):
                xx = torch.flip(x, fl) if fl else x
                with torch.autocast("cuda", dtype=torch.float16):
                    p = torch.sigmoid(model(xx.contiguous(memory_format=torch.channels_last)).float())
                p = torch.flip(p, fl) if fl else p
                acc = acc + (p.transpose(2, 3) if tr else p)
        return (acc[0] / 8 * 255).round().byte().cpu().numpy()

    def dump(zpath, names, arrays_iter):
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
            for name, p in zip(names, arrays_iter):
                ok, buf = cv2.imencode(".png", np.dstack([p[0], p[1], np.zeros_like(p[0])]))
                assert ok
                z.writestr(name + ".png", buf.tobytes())

    def run_fold(f, smoke=False):
        tag = f"smoke{gpu}" if smoke else f"fold{f}"
        tr = np.flatnonzero(fold_of != f); va = np.flatnonzero(fold_of == f)
        torch.manual_seed(CFG["seed"] + f)
        model = smp.Unet(CFG["enc"], encoder_weights="imagenet", in_channels=1, classes=2)
        model = model.cuda().to(memory_format=torch.channels_last)
        ema = copy.deepcopy(model).eval()
        for p in ema.parameters():
            p.requires_grad_(False)
        dl = torch.utils.data.DataLoader(
            DS(tr), batch_size=CFG["bs"], shuffle=True, drop_last=True, num_workers=CFG["loader_workers"],
            pin_memory=True, persistent_workers=True,
            worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2 ** 31))
        total = 3 if smoke else CFG["epochs"] * len(dl)
        warm = min(300, total // 10 + 1)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
        scaler = torch.amp.GradScaler("cuda")
        step, run, t0 = 0, 0.0, time.time()
        model.train()
        while step < total:
            for x, y in dl:
                x = norm(x.cuda(non_blocking=True)).contiguous(memory_format=torch.channels_last)
                y = y.cuda(non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    logit = model(x).float()
                bce = F.binary_cross_entropy_with_logits(logit, y)
                p = torch.sigmoid(logit)
                dice = 1 - ((2 * (p * y).sum((0, 2, 3)) + 1) / (p.sum((0, 2, 3)) + y.sum((0, 2, 3)) + 1)).mean()
                loss = bce + 0.5 * dice
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); sched.step()
                d = min(CFG["ema"], (1 + step) / (10 + step))
                with torch.no_grad():
                    for pe, pm in zip(ema.parameters(), model.parameters()):
                        pe.lerp_(pm, 1 - d)
                    for be, bm in zip(ema.buffers(), model.buffers()):
                        be.copy_(bm)
                step += 1
                run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
                if step % 200 == 0:
                    log(f"{tag} step {step}/{total} loss {run:.4f} bce {bce.item():.4f} "
                        f"{(time.time() - t0) / step:.2f}s/it")
                if step >= total:
                    break
        del dl

        imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
        tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
        test = np.load(f"{CACHE}/test.npy", mmap_mode="r")
        va_use = va[:2] if smoke else va
        te_use = range(2 if smoke else len(test))
        inter = np.zeros(2); den = np.zeros(2)

        def oof_iter():
            for j in va_use:
                p = predict(ema, imgs[j])
                t = np.asarray(tgt[j]) >= 128
                q = p >= 128
                inter[:] += (q & t).sum((1, 2)); den[:] += q.sum((1, 2)) + t.sum((1, 2))
                yield p

        dump(f"{OUT}/oof_{tag}.zip", [meta["stems"][j] for j in va_use], oof_iter())
        log(f"{tag} OOF pixel dice @0.5: shape {2 * inter[0] / max(den[0], 1):.4f} "
            f"presence {2 * inter[1] / max(den[1], 1):.4f}")
        dump(f"{OUT}/test_{tag}.zip", [meta["test_stems"][j] for j in te_use], (predict(ema, test[j]) for j in te_use))
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in ema.state_dict().items()},
                   f"{OUT}/model_{tag}.pt")
        if smoke:
            for fn in (f"oof_{tag}.zip", f"test_{tag}.zip", f"model_{tag}.pt"):
                os.remove(f"{OUT}/{fn}")
        log(f"{tag} done")
        del model, ema, opt
        torch.cuda.empty_cache()

    log(f"worker gpu {gpu} folds {folds}: {torch.cuda.get_device_name(0)}")
    run_fold(folds[0], smoke=True)      # end-to-end check of train / predict / save before spending GPU time
    for f in folds:
        run_fold(f)


def main():
    # The parent imports neither torch nor cv2, so the forked children start from a clean process.
    import importlib.util
    for mod, pkg in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("pycocotools", "pycocotools")):
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    assert n_gpu >= 1, "no GPU"
    log("gpus", n_gpu, CFG)
    ctx = mp.get_context("fork")
    procs = [ctx.Process(target=prepare)]
    procs[0].start(); procs[0].join()
    assert procs[0].exitcode == 0, "prepare failed"
    procs = [ctx.Process(target=worker, args=(g, list(range(g, CFG["folds"], n_gpu)))) for g in range(n_gpu)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    assert all(p.exitcode == 0 for p in procs), [p.exitcode for p in procs]
    log("all done", sorted(os.listdir(OUT)))


main()


In [ ]:
%%writefile /kaggle/working/training_source/train_v3.py
"""Solar filament segmentation - grouped K-fold training of a 2-channel soft-label U-Net.

Targets (both in [0, 1], built from the per-annotator polygons):
  ch0 "shape":    (#annotators covering the pixel) / (#annotators who drew anything in this filament cluster)
  ch1 "presence": (#annotators covering the pixel) / (#annotators of the image)
A filament cluster is an 8-connected component of the union of all annotators' masks.

Outputs (in /kaggle/working): oof_fold{k}.zip, test_fold{k}.zip (PNG probability maps, uint8, B=ch0 G=ch1),
model_fold{k}.pt, meta.json. Folds are trained in parallel, one process per GPU.
"""
import glob
import io
import json
import math
import multiprocessing as mp
import os
import subprocess
import sys
import time
import zipfile

import numpy as np

CFG = dict(folds=4, crop=768, bs=8, epochs=36, crops_per_img=3, lr=3e-4, wd=1e-4, enc="efficientnet-b4",
           seed=42, ema=0.998, loader_workers=2)
CACHE = "/kaggle/temp/cache"
OUT = "/kaggle/working"
H = W = 2048
T0 = time.time()


def log(*a):
    print(f"[{(time.time() - T0) / 60:6.1f}m]", *a, flush=True)


# ----------------------------------------------------------------------------- data cache (CPU, parent process)
def prepare():
    import cv2
    from pycocotools import mask as mu

    os.makedirs(CACHE, exist_ok=True)
    js = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
    root = os.path.dirname(os.path.dirname(js))
    train_dir = os.path.join(root, "train", "train_images")
    test_dir = os.path.join(root, "test", "test_images")
    coco = json.load(open(js))
    log("labels", js)

    sets_of, anns_of = {}, {}
    for im in coco["images"]:
        sets_of.setdefault(os.path.splitext(im["file_name"])[0], []).append(im["id"])
    for a in coco["annotations"]:
        anns_of.setdefault(a["image_id"], []).append(a)
    stems = sorted(sets_of)
    test_stems = sorted(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(test_dir, "*.jpeg")))
    n = len(stems)
    log(f"{n} train images, {len(coco['images'])} annotator-sets, {len(test_stems)} test images")

    imgs = np.lib.format.open_memmap(f"{CACHE}/imgs.npy", "w+", np.uint8, (n, H, W))
    tgt = np.lib.format.open_memmap(f"{CACHE}/tgt.npy", "w+", np.uint8, (n, 2, H, W))
    centers = []
    for i, s in enumerate(stems):
        im = cv2.imread(os.path.join(train_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), (s, None if im is None else im.shape)
        imgs[i] = im
        per_annot, cs = [], []
        for aset in sets_of[s]:
            m = np.zeros((H, W), np.uint8)
            for a in anns_of.get(aset, []):
                m |= mu.decode(mu.merge(mu.frPyObjects(a["segmentation"], H, W)))
                x, y, w, h = a["bbox"]
                cs.append([x + w / 2, y + h / 2])
            per_annot.append(m)
        count = np.sum(per_annot, axis=0).astype(np.float32)
        ncc, lab = cv2.connectedComponents((count > 0).astype(np.uint8), connectivity=8)
        k = np.zeros(ncc, np.float32)
        for m in per_annot:
            k[np.unique(lab[m > 0])] += 1
        k[0] = 1
        tgt[i, 0] = np.rint(255 * count / np.maximum(k[lab], 1)).astype(np.uint8)
        tgt[i, 1] = np.rint(255 * count / len(per_annot)).astype(np.uint8)
        centers.append(cs)
        if i % 100 == 0:
            log("cache train", i)
    imgs.flush(); tgt.flush(); del imgs, tgt

    test = np.lib.format.open_memmap(f"{CACHE}/test.npy", "w+", np.uint8, (len(test_stems), H, W))
    for i, s in enumerate(test_stems):
        im = cv2.imread(os.path.join(test_dir, s + ".jpeg"), cv2.IMREAD_GRAYSCALE)
        assert im is not None and im.shape == (H, W), s
        test[i] = im
    test.flush(); del test

    perm = np.random.RandomState(CFG["seed"]).permutation(n)
    fold = np.zeros(n, int)
    fold[perm] = np.arange(n) % CFG["folds"]
    meta = dict(stems=stems, test_stems=test_stems, fold=fold.tolist(), centers=centers,
                n_sets=[len(sets_of[s]) for s in stems], cfg=CFG)
    json.dump(meta, open(f"{CACHE}/meta.json", "w"))
    json.dump({k: v for k, v in meta.items() if k != "centers"}, open(f"{OUT}/meta.json", "w"))
    log("cache done")


# ----------------------------------------------------------------------------- worker (one per GPU)
def worker(gpu, folds):
    os.environ["CUDA_VISIBLE_DEVICES"] = str(gpu)
    import copy
    import cv2
    import torch
    import torch.nn.functional as F
    import segmentation_models_pytorch as smp

    cv2.setNumThreads(0)
    torch.backends.cudnn.benchmark = True
    meta = json.load(open(f"{CACHE}/meta.json"))
    fold_of = np.array(meta["fold"])
    C = CFG["crop"]

    class DS(torch.utils.data.Dataset):
        def __init__(self, idx):
            self.idx, self.imgs, self.tgt = idx, None, None

        def __len__(self):
            return len(self.idx) * CFG["crops_per_img"]

        def __getitem__(self, i):
            if self.imgs is None:
                self.imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
                self.tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
            j = int(self.idx[i % len(self.idx)])
            r = np.random
            win = int(round(C / math.exp(r.uniform(math.log(0.7), math.log(1.4)))))
            cs = meta["centers"][j]
            if cs and r.rand() < 0.6:
                cx, cy = cs[r.randint(len(cs))]
                cx += r.uniform(-win / 3, win / 3); cy += r.uniform(-win / 3, win / 3)
            else:
                cx, cy = r.uniform(0, W), r.uniform(0, H)
            x0 = int(np.clip(cx - win / 2, 0, W - win)); y0 = int(np.clip(cy - win / 2, 0, H - win))
            im = np.array(self.imgs[j, y0:y0 + win, x0:x0 + win])
            tg = np.array(self.tgt[j, :, y0:y0 + win, x0:x0 + win])
            if win != C:
                interp = cv2.INTER_AREA if win > C else cv2.INTER_LINEAR
                im = cv2.resize(im, (C, C), interpolation=interp)
                tg = np.stack([cv2.resize(t, (C, C), interpolation=cv2.INTER_LINEAR) for t in tg])
            k = r.randint(4)
            im, tg = np.rot90(im, k), np.rot90(tg, k, axes=(1, 2))
            if r.rand() < 0.5:
                im, tg = im[:, ::-1], tg[:, :, ::-1]
            x = im.astype(np.float32) / 255.0
            x = np.clip((x - 0.5) * r.uniform(0.8, 1.25) + 0.5 + r.uniform(-0.08, 0.08), 0, 1) ** r.uniform(0.8, 1.25)
            return (torch.from_numpy(np.ascontiguousarray(x))[None],
                    torch.from_numpy(np.ascontiguousarray(tg)).float() / 255.0)

    def norm(x):
        return (x - 0.45) / 0.225

    @torch.no_grad()
    def predict(model, im):
        """uint8 (H, W) -> uint8 (2, H, W) probabilities, 8-view dihedral TTA."""
        x0 = norm(torch.from_numpy(np.ascontiguousarray(im)).cuda().float()[None, None] / 255.0)
        acc = 0
        for tr in (False, True):
            x = x0.transpose(2, 3) if tr else x0
            for fl in ([], [2], [3], [2, 3]):
                xx = torch.flip(x, fl) if fl else x
                with torch.autocast("cuda", dtype=torch.float16):
                    p = torch.sigmoid(model(xx.contiguous(memory_format=torch.channels_last)).float())
                p = torch.flip(p, fl) if fl else p
                acc = acc + (p.transpose(2, 3) if tr else p)
        return (acc[0] / 8 * 255).round().byte().cpu().numpy()

    def dump(zpath, names, arrays_iter):
        with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
            for name, p in zip(names, arrays_iter):
                ok, buf = cv2.imencode(".png", np.dstack([p[0], p[1], np.zeros_like(p[0])]))
                assert ok
                z.writestr(name + ".png", buf.tobytes())

    def run_fold(f, smoke=False):
        tag = f"smoke{gpu}" if smoke else f"fold{f}"
        tr = np.flatnonzero(fold_of != f); va = np.flatnonzero(fold_of == f)
        torch.manual_seed(1003 + f); np.random.seed(1003 + f)
        model = smp.Unet(CFG["enc"], encoder_weights="imagenet", in_channels=1, classes=2)
        model = model.cuda().to(memory_format=torch.channels_last)
        ema = copy.deepcopy(model).eval()
        for p in ema.parameters():
            p.requires_grad_(False)
        dl = torch.utils.data.DataLoader(
            DS(tr), batch_size=CFG["bs"], shuffle=True, drop_last=True, num_workers=CFG["loader_workers"],
            pin_memory=True, persistent_workers=True,
            worker_init_fn=lambda w: np.random.seed((torch.initial_seed() + w) % 2 ** 31))
        total = 3 if smoke else CFG["epochs"] * len(dl)
        warm = min(300, total // 10 + 1)
        opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["wd"])
        sched = torch.optim.lr_scheduler.LambdaLR(
            opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, total - warm))))
        scaler = torch.amp.GradScaler("cuda")
        step, run, t0 = 0, 0.0, time.time()
        model.train()
        while step < total:
            for x, y in dl:
                x = norm(x.cuda(non_blocking=True)).contiguous(memory_format=torch.channels_last)
                y = y.cuda(non_blocking=True)
                with torch.autocast("cuda", dtype=torch.float16):
                    logit = model(x).float()
                bce = F.binary_cross_entropy_with_logits(logit, y)
                p = torch.sigmoid(logit)
                dice = 1 - ((2 * (p * y).sum((0, 2, 3)) + 1) / (p.sum((0, 2, 3)) + y.sum((0, 2, 3)) + 1)).mean()
                loss = bce + 0.5 * dice
                opt.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                scaler.step(opt); scaler.update(); sched.step()
                d = min(CFG["ema"], (1 + step) / (10 + step))
                with torch.no_grad():
                    for pe, pm in zip(ema.parameters(), model.parameters()):
                        pe.lerp_(pm, 1 - d)
                    for be, bm in zip(ema.buffers(), model.buffers()):
                        be.copy_(bm)
                step += 1
                run = 0.98 * run + 0.02 * loss.item() if step > 1 else loss.item()
                if step % 200 == 0:
                    log(f"{tag} step {step}/{total} loss {run:.4f} bce {bce.item():.4f} "
                        f"{(time.time() - t0) / step:.2f}s/it")
                if step >= total:
                    break
        del dl

        imgs = np.load(f"{CACHE}/imgs.npy", mmap_mode="r")
        tgt = np.load(f"{CACHE}/tgt.npy", mmap_mode="r")
        test = np.load(f"{CACHE}/test.npy", mmap_mode="r")
        va_use = va[:2] if smoke else va
        te_use = range(2 if smoke else len(test))
        inter = np.zeros(2); den = np.zeros(2)

        def oof_iter():
            for j in va_use:
                p = predict(ema, imgs[j])
                t = np.asarray(tgt[j]) >= 128
                q = p >= 128
                inter[:] += (q & t).sum((1, 2)); den[:] += q.sum((1, 2)) + t.sum((1, 2))
                yield p

        dump(f"{OUT}/oof_{tag}.zip", [meta["stems"][j] for j in va_use], oof_iter())
        log(f"{tag} OOF pixel dice @0.5: shape {2 * inter[0] / max(den[0], 1):.4f} "
            f"presence {2 * inter[1] / max(den[1], 1):.4f}")
        dump(f"{OUT}/test_{tag}.zip", [meta["test_stems"][j] for j in te_use], (predict(ema, test[j]) for j in te_use))
        torch.save({k: v.half() if v.is_floating_point() else v for k, v in ema.state_dict().items()},
                   f"{OUT}/model_{tag}.pt")
        if smoke:
            for fn in (f"oof_{tag}.zip", f"test_{tag}.zip", f"model_{tag}.pt"):
                os.remove(f"{OUT}/{fn}")
        log(f"{tag} done")
        del model, ema, opt
        torch.cuda.empty_cache()

    log(f"worker gpu {gpu} folds {folds}: {torch.cuda.get_device_name(0)}")
    run_fold(folds[0], smoke=True)      # end-to-end check of train / predict / save before spending GPU time
    for f in folds:
        run_fold(f)


def main():
    # The parent imports neither torch nor cv2, so the forked children start from a clean process.
    import importlib.util
    for mod, pkg in (("segmentation_models_pytorch", "segmentation-models-pytorch"), ("pycocotools", "pycocotools")):
        if importlib.util.find_spec(mod) is None:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
    n_gpu = len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())
    assert n_gpu >= 1, "no GPU"
    log("gpus", n_gpu, CFG)
    ctx = mp.get_context("fork")
    procs = [ctx.Process(target=prepare)]
    procs[0].start(); procs[0].join()
    assert procs[0].exitcode == 0, "prepare failed"
    procs = [ctx.Process(target=worker, args=(g, list(range(g, CFG["folds"], n_gpu)))) for g in range(n_gpu)]
    for p in procs:
        p.start()
    for p in procs:
        p.join()
    assert all(p.exitcode == 0 for p in procs), [p.exitcode for p in procs]
    log("all done", sorted(os.listdir(OUT)))


main()


In [ ]:
%%writefile /kaggle/working/training_source/train_yolo.py
"""YOLO instance segmentation for solar filaments (detect-then-segment), holdout = fold HOLD of the shared split.
Writes pred_hold.pkl / pred_test.pkl: stem -> list of (score, RLE counts str) at 2048 x 2048, plus weights.
"""
import glob, json, os, pickle, subprocess, sys, time
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics", "pycocotools"])
import numpy as np, cv2, torch
from pycocotools import mask as mu
from ultralytics import YOLO

FOLD = {"20110109104734Ch": 2, "20110114105034Ch": 1, "20110119082634Lh": 3, "20110123174914Mh": 3, "20110128174814Mh": 2, "20110203082634Lh": 1, "20110205082634Lh": 0, "20110211084114Th": 1, "20110213093414Th": 2, "20110215083814Th": 1, "20110220082634Lh": 1, "20110221173814Mh": 2, "20110301082654Uh": 2, "20110306082634Lh": 0, "20110312082634Lh": 2, "20110313082654Uh": 2, "20110317082654Uh": 3, "20110322005814Mh": 2, "20110324082614Th": 1, "20110325171014Mh": 1, "20110326170914Mh": 0, "20110327143554Bh": 1, "20110406082634Lh": 3, "20110410141654Bh": 1, "20110421164914Mh": 1, "20110425063154Uh": 0, "20110429135554Bh": 1, "20110502063154Uh": 1, "20110509121334Ch": 2, "20110510063134Lh": 1, "20110511063154Uh": 2, "20110517164514Mh": 3, "20110520063154Uh": 2, "20110521063134Lh": 1, "20110523063154Uh": 2, "20110525063154Uh": 0, "20110526122534Ch": 0, "20110530122834Ch": 3, "20110601063134Lh": 3, "20110603163414Mh": 1, "20110615133454Bh": 0, "20110616133454Bh": 3, "20110617063134Lh": 0, "20110619063134Lh": 3, "20110621063134Lh": 2, "20110623063134Lh": 3, "20110625163814Mh": 1, "20110707074514Th": 1, "20110711071214Th": 0, "20110724164614Mh": 2, "20110727063234Lh": 0, "20110728084714Th": 1, "20110803164914Mh": 1, "20110810063134Lh": 1, "20110815140454Bh": 0, "20110823120334Ch": 2, "20110824141054Bh": 2, "20110826074214Th": 2, "20110902073614Th": 1, "20110914063134Lh": 0, "20110918112834Ch": 0, "20110923112234Ch": 2, "20110923165814Mh": 0, "20111009110234Ch": 2, "20111011112054Uh": 1, "20111018063134Lh": 0, "20111019063154Uh": 1, "20111025080614Th": 2, "20111027145954Bh": 3, "20111029063154Uh": 3, "20111107063154Uh": 0, "20111112193254Bh": 2, "20111114082314Th": 2, "20111115063134Lh": 1, "20111116063134Lh": 0, "20111201063154Uh": 1, "20111206102934Ch": 2, "20111208063134Lh": 2, "20111214063134Lh": 3, "20111214132434Ch": 0, "20111215063134Lh": 0, "20111218085014Th": 1, "20111222063154Uh": 0, "20111223085314Th": 2, "20111226155454Bh": 3, "20111229063154Uh": 0, "20120103104134Ch": 1, "20120104063134Lh": 1, "20120110155754Bh": 0, "20120113104834Ch": 2, "20120114063154Uh": 1, "20120116063254Uh": 3, "20120127085214Th": 0, "20120131063154Uh": 1, "20120205172414Mh": 0, "20120206063134Lh": 2, "20120213063134Lh": 1, "20120215153654Bh": 0, "20120217063154Uh": 1, "20120217181154Bh": 3, "20120218173614Mh": 2, "20120223063154Uh": 2, "20120223152054Bh": 2, "20120225172914Mh": 0, "20120227161654Bh": 3, "20120306063154Uh": 2, "20120313063154Uh": 3, "20120318063154Uh": 2, "20120323144054Bh": 0, "20120328164614Mh": 1, "20120330063134Lh": 3, "20120330063154Uh": 1, "20120331142954Bh": 3, "20120403083114Th": 2, "20120403164114Mh": 0, "20120404063154Uh": 1, "20120414115434Ch": 0, "20120414175914Mh": 2, "20120421063154Uh": 3, "20120424072414Th": 0, "20120424120134Ch": 1, "20120427063154Uh": 2, "20120427162614Mh": 1, "20120430063154Uh": 1, "20120509163414Mh": 3, "20120519070814Th": 3, "20120523163014Mh": 2, "20120603123034Ch": 0, "20120610175114Mh": 2, "20120627162114Mh": 0, "20120703070814Th": 1, "20120712123634Ch": 3, "20120715123434Ch": 1, "20120720085114Th": 0, "20120723095014Th": 0, "20120725122934Ch": 0, "20120726071914Th": 0, "20120729122634Ch": 2, "20120730135454Bh": 3, "20120802090314Th": 2, "20120803102114Th": 1, "20120803164614Mh": 2, "20120810121634Ch": 1, "20120811121434Ch": 3, "20120813164714Mh": 2, "20120814140454Bh": 0, "20120814163114Mh": 0, "20120816072814Th": 3, "20120827085114Th": 3, "20120829063134Lh": 2, "20120901114934Ch": 2, "20120907141954Bh": 2, "20120910074014Th": 2, "20120913165314Mh": 0, "20120914063134Lh": 1, "20120917142654Bh": 2, "20120928063134Lh": 3, "20121006063534Lh": 0, "20121007131134Ch": 2, "20121009165914Mh": 2, "20121018063134Lh": 0, "20121021145454Bh": 2, "20121022145554Bh": 2, "20121024164714Mh": 0, "20121026063134Lh": 1, "20121027063134Lh": 3, "20121027165114Mh": 1, "20121030150354Bh": 2, "20121108155954Bh": 0, "20121109103634Ch": 2, "20121113103634Ch": 2, "20121114172254Bh": 2, "20121118092514Th": 2, "20121129063154Uh": 0, "20121207103034Ch": 0, "20121208171514Mh": 1, "20121218063534Lh": 2, "20121219063154Uh": 3, "20130107174854Bh": 1, "20130109130334Ch": 2, "20130110104734Ch": 1, "20130111104734Ch": 2, "20130116172814Mh": 3, "20130201084914Th": 2, "20130208063154Uh": 1, "20130211063134Lh": 2, "20130212153254Bh": 0, "20130214111534Ch": 3, "20130218220314Mh": 1, "20130219063134Lh": 2, "20130221063134Lh": 0, "20130222082914Th": 2, "20130225171214Mh": 0, "20130303063134Lh": 0, "20130303063154Uh": 1, "20130304170614Mh": 3, "20130305112734Ch": 1, "20130306172214Mh": 1, "20130314165714Mh": 1, "20130319144654Bh": 2, "20130324015554Uh": 3, "20130326063134Lh": 1, "20130327175114Mh": 3, "20130404063134Lh": 3, "20130405114734Ch": 3, "20130407063154Uh": 1, "20130411115134Ch": 1, "20130419070254Uh": 1, "20130423072414Th": 3, "20130425135954Bh": 1, "20130429063154Uh": 3, "20130502063134Lh": 1, "20130510134654Bh": 3, "20130517063134Lh": 0, "20130601063154Uh": 0, "20130609123334Ch": 0, "20130609173414Mh": 0, "20130611063154Uh": 1, "20130615002434Lh": 3, "20130616070414Th": 0, "20130628063134Lh": 1, "20130630070714Th": 1, "20130705161714Mh": 0, "20130707134154Bh": 3, "20130716162214Mh": 1, "20130717123434Ch": 0, "20130719140154Bh": 1, "20130725122934Ch": 1, "20130805135854Bh": 1, "20130807073114Th": 2, "20130808140054Bh": 3, "20130809072814Th": 2, "20130810121534Ch": 0, "20130811140254Bh": 3, "20130813060834Lh": 3, "20130824060834Lh": 1, "20130828141354Bh": 3, "20130901073614Th": 1, "20130903060834Lh": 1, "20130903141654Bh": 3, "20130905073714Th": 0, "20130913142354Bh": 3, "20130919060834Lh": 1, "20130921060834Lh": 3, "20130923060834Lh": 2, "20130925143254Bh": 1, "20130926060834Lh": 1, "20131006075314Th": 1, "20131007075314Th": 2, "20131011195414Mh": 2, "20131013185334Ch": 3, "20131016185334Ch": 0, "20131017185334Ch": 0, "20131019080114Th": 3, "20131020080114Th": 0, "20131025185414Mh": 1, "20131027025654Uh": 2, "20131028190734Ch": 1, "20131030185354Bh": 1, "20131031185614Mh": 1, "20131101222734Lh": 0, "20131107043054Uh": 2, "20131108045254Uh": 1, "20131109190514Mh": 3, "20131109222434Lh": 2, "20131113004514Mh": 2, "20131113185414Mh": 3, "20131115185414Mh": 0, "20131116185334Ch": 3, "20131125192154Bh": 0, "20131128192214Mh": 0, "20131130192134Ch": 2, "20131201192134Ch": 0, "20131204192154Bh": 0, "20131206084114Th": 0, "20131207084214Th": 0, "20131224192154Bh": 1, "20131225065154Uh": 1, "20131229112414Th": 3, "20140111224134Lh": 2, "20140112180134Ch": 1, "20140116180214Mh": 0, "20140122180214Mh": 2, "20140127180134Ch": 0, "20140128174554Bh": 1, "20140205180154Bh": 2, "20140210180154Bh": 2, "20140212024554Uh": 1, "20140213024554Uh": 2, "20140216230134Lh": 2, "20140218230234Lh": 0, "20140225163314Th": 0, "20140312025454Uh": 3, "20140312081114Th": 3, "20140321195814Mh": 1, "20140323020654Uh": 3, "20140401014754Uh": 1, "20140406195854Bh": 3, "20140409081614Th": 2, "20140411013854Uh": 3, "20140413195914Mh": 3, "20140414020354Uh": 2, "20140414195834Ch": 1, "20140420085614Th": 0, "20140428233234Lh": 3, "20140430071914Th": 3, "20140501195814Mh": 0, "20140508195834Ch": 0, "20140509071314Th": 3, "20140509195814Mh": 3, "20140513012554Uh": 0, "20140522070614Th": 3, "20140523195814Mh": 0, "20140525070514Th": 2, "20140531195834Ch": 0, "20140601195814Mh": 1, "20140603070414Th": 2, "20140604195854Bh": 1, "20140605195854Bh": 1, "20140607112354Uh": 0, "20140607195854Bh": 0, "20140609195854Bh": 3, "20140615195814Mh": 0, "20140615195854Bh": 2, "20140701195814Mh": 3, "20140705035754Uh": 3, "20140709195814Mh": 1, "20140710083214Th": 0, "20140712195854Bh": 3, "20140716195854Bh": 2, "20140719071414Th": 3, "20140720195834Ch": 1, "20140723071614Th": 1, "20140725195814Mh": 0, "20140726071814Th": 1, "20140802072114Th": 0, "20140803234534Lh": 0, "20140806195834Ch": 2, "20140807195854Bh": 1, "20140813064254Uh": 0, "20140814195854Bh": 0, "20140816072814Th": 0, "20140817195834Ch": 1, "20140818195814Mh": 2, "20140821195834Ch": 2, "20140903073614Th": 2, "20140904195814Mh": 1, "20140908175354Bh": 1, "20140908231334Lh": 1, "20140918195854Bh": 1, "20140921195854Bh": 2, "20140922195854Bh": 2, "20140927014554Uh": 2, "20141001195834Ch": 2, "20141004075414Th": 0, "20141006224634Lh": 3, "20141007224934Lh": 3, "20141009023454Uh": 0, "20141010195834Ch": 0, "20141012015654Uh": 1, "20141012195854Bh": 3, "20141013195834Ch": 3, "20141013195854Bh": 2, "20141016022554Uh": 1, "20141016195834Ch": 1, "20141022223334Lh": 2, "20141024195834Ch": 2, "20141025031154Uh": 3, "20141026024454Uh": 3, "20141027025054Uh": 3, "20141027224834Lh": 0, "20141105195814Mh": 1, "20141106195854Bh": 0, "20141112195854Bh": 2, "20141116222234Lh": 3, "20141129222134Lh": 2, "20141210201814Mh": 3, "20141214201834Ch": 0, "20141216222634Lh": 1, "20141225024354Uh": 2, "20141225172214Mh": 3, "20150112081954Uh": 3, "20150113105034Ch": 3, "20150113155654Bh": 3, "20150116082034Lh": 0, "20150119082034Lh": 3, "20150125172714Mh": 1, "20150130084914Th": 3, "20150203084714Th": 3, "20150207084414Th": 0, "20150216171714Mh": 0, "20150221155154Bh": 0, "20150222090014Th": 2, "20150222112034Ch": 3, "20150225171114Mh": 2, "20150228082414Th": 3, "20150302082034Lh": 2, "20150305081954Uh": 3, "20150308081954Uh": 0, "20150314081954Uh": 3, "20150315082034Lh": 0, "20150323172814Mh": 0, "20150324145954Bh": 1, "20150325015554Uh": 3, "20150327145114Th": 0, "20150330081954Uh": 1, "20150409081954Uh": 3, "20150417082314Th": 2, "20150421140454Bh": 0, "20150423120034Ch": 0, "20150427162314Mh": 2, "20150501120634Ch": 2, "20150518145654Bh": 0, "20150527082014Th": 0, "20150527161214Mh": 1, "20150531082034Lh": 2, "20150610082014Th": 2, "20150614162514Mh": 3, "20150615133554Bh": 3, "20150617161314Mh": 0, "20150620082014Th": 2, "20150621161314Mh": 1, "20150625082014Th": 1, "20150625082034Lh": 1, "20150626161514Mh": 0, "20150627133754Bh": 1, "20150629161614Mh": 2, "20150630083214Th": 0, "20150706083014Th": 2, "20150716162114Mh": 1, "20150718082014Th": 2, "20150719182314Mh": 3, "20150720082014Th": 0, "20150727082014Th": 3, "20150731082034Lh": 0, "20150806082034Lh": 1, "20150819082014Th": 1, "20150820152254Bh": 3, "20150825082034Lh": 0, "20150830084134Lh": 0, "20150913082014Th": 1, "20150916055454Uh": 1, "20150919165814Mh": 1, "20150926111934Ch": 3, "20150927111834Ch": 3, "20151011144554Bh": 3, "20151013144654Bh": 3, "20151019223534Lh": 2, "20151105133734Ch": 3, "20151106171514Mh": 1, "20151107133734Ch": 1, "20151116201114Mh": 3, "20151116224034Lh": 1, "20151117172314Mh": 3, "20151118172314Mh": 1, "20151119133734Ch": 2, "20151122172614Mh": 3, "20151128133714Th": 1, "20151128133734Ch": 2, "20151128222134Lh": 3, "20151209201754Bh": 2, "20151212171154Bh": 1, "20151213023454Uh": 0, "20151219133734Ch": 2, "20151224133734Ch": 1, "20151226133734Ch": 1, "20151228024254Uh": 1, "20160101133714Th": 2, "20160103133734Ch": 2, "20160107133714Th": 0, "20160110133714Th": 3, "20160125154854Bh": 1, "20160127225034Lh": 0, "20160129225334Lh": 0, "20160201025254Uh": 2, "20160204133734Ch": 3, "20160213153054Bh": 1, "20160215133734Ch": 0, "20160215175514Mh": 0, "20160223173514Mh": 0, "20160225173314Mh": 0, "20160226173214Mh": 3, "20160229151254Bh": 3, "20160302133734Ch": 0, "20160309172214Mh": 0, "20160310133734Ch": 0, "20160315133714Th": 0, "20160317232434Lh": 3, "20160320133714Th": 3, "20160326143654Bh": 0, "20160327170714Mh": 1, "20160401133734Ch": 1, "20160401170314Mh": 1, "20160402025254Uh": 3, "20160403170114Mh": 3, "20160408232234Lh": 1, "20160411134234Ch": 2, "20160430133714Th": 2, "20160430133734Ch": 2, "20160501143154Bh": 2, "20160510163914Mh": 3, "20160516012454Uh": 0, "20160518011654Uh": 0, "20160526011254Uh": 2, "20160530234734Lh": 2, "20160604163414Mh": 2, "20160606163514Mh": 0, "20160609133654Bh": 3, "20160617133654Bh": 3, "20160619133714Th": 3, "20160627133734Ch": 0, "20160628152654Bh": 0, "20160629133714Th": 1, "20160703133714Th": 2, "20160714235434Lh": 1, "20160720133734Ch": 3, "20160724135554Bh": 0, "20160726135354Bh": 2, "20160804133734Ch": 0, "20160808133714Th": 1, "20160809164914Mh": 0, "20160810234034Lh": 0, "20160831133714Th": 3, "20160920230134Lh": 3, "20160925165814Mh": 3, "20160925225634Lh": 1, "20160927133734Ch": 3, "20160928143554Bh": 0, "20161002225034Lh": 1, "20161012133734Ch": 3, "20161013133734Ch": 1, "20161023133814Th": 3, "20161105222634Lh": 1, "20161108021454Uh": 2, "20161109171714Mh": 3, "20161110151454Bh": 3, "20161116133714Th": 2, "20161129022454Uh": 0, "20161201160954Bh": 0, "20161205133714Th": 1, "20161210133714Th": 1, "20161217133734Ch": 1, "20161222133714Th": 2, "20161223133734Ch": 0, "20161224024154Uh": 1, "20161229133734Ch": 0, "20161231133734Ch": 0, "20170102024454Uh": 1, "20170103030554Uh": 3, "20170106133714Th": 2, "20170107133714Th": 3, "20170117175014Mh": 1, "20170119031254Uh": 0, "20170122034654Uh": 3, "20170125133734Ch": 2, "20170217165250Bh": 3, "20170218173710Mh": 3, "20170222133710Th": 0, "20170310133730Ch": 0, "20170310173450Bh": 3, "20170316133730Ch": 3, "20170322015550Uh": 3, "20170323041730Lh": 1, "20170327143550Bh": 3, "20170329133730Ch": 1, "20170408133710Th": 3, "20170411013550Uh": 0, "20170415145450Bh": 2, "20170417014150Uh": 2, "20170417232530Lh": 1, "20170418013450Uh": 3, "20170423014350Uh": 3, "20170425133730Ch": 3, "20170427154550Bh": 2, "20170502233130Lh": 1, "20170503013050Uh": 3, "20170609133710Th": 2, "20170612135150Bh": 3, "20170619012150Uh": 0, "20170701235630Lh": 3, "20170702133730Ch": 1, "20170719235330Lh": 0, "20170722235230Lh": 3, "20170726235030Lh": 0, "20170728150230Ch": 1, "20170801164710Mh": 2, "20170804140650Bh": 3, "20170805133710Th": 0, "20170806133710Th": 2, "20170809164910Mh": 2, "20170827165310Mh": 1, "20170831170750Bh": 2, "20170925225630Lh": 1, "20170927143450Bh": 2, "20171004143950Bh": 2, "20171005133730Ch": 0, "20171005144050Bh": 0, "20171007133710Th": 1, "20171007133730Ch": 2, "20171008144250Bh": 2, "20171011170310Mh": 2, "20171011224130Lh": 2, "20171014133730Ch": 0, "20171022170710Mh": 3, "20171025133730Ch": 3, "20171025223130Lh": 2, "20171027223030Lh": 1, "20171029150150Bh": 2, "20171101133730Ch": 3, "20171103222630Lh": 2, "20171106171510Mh": 1, "20171109023950Uh": 3, "20171114133730Ch": 1, "20171115170950Bh": 1, "20171115222230Lh": 3, "20171121222130Lh": 1, "20171123133730Ch": 0, "20171123222130Lh": 0, "20180109223930Lh": 0, "20180112130330Ch": 2, "20180202130330Ch": 2, "20180301050430Lh": 3, "20180306081810Th": 3, "20180309000910Mh": 2, "20180318020350Uh": 3, "20180318232230Lh": 3, "20180506140230Ch": 2, "20180710031950Uh": 1, "20180712123630Ch": 2, "20180714000010Mh": 3, "20180801232450Bh": 3, "20180804234530Lh": 0, "20180805232510Mh": 1, "20180825205030Ch": 0, "20180827232530Lh": 1, "20180923112330Ch": 1, "20180924112030Ch": 0, "20180925000030Lh": 0, "20180926014350Uh": 1, "20181014105830Ch": 3, "20181023232430Lh": 0, "20181024232430Lh": 1, "20181101230010Mh": 0, "20181102104630Ch": 2, "20190317145150Bh": 3, "20190322144450Bh": 2, "20190412025430Lh": 3, "20190602025430Lh": 0, "20190609123330Ch": 2, "20190611032810Mh": 1, "20190626025430Lh": 3, "20190629163710Mh": 0, "20190701123830Ch": 0, "20190704123830Ch": 3, "20190805054730Lh": 2, "20190925165810Mh": 2, "20190926165810Mh": 2, "20190927165910Mh": 1, "20200105073250Uh": 2, "20200110073230Lh": 2, "20200114073230Lh": 2, "20200116073250Uh": 0, "20200619123010Th": 3, "20200806053830Lh": 2, "20201022223330Lh": 1, "20201227202710Mh": 0, "20201228202430Ch": 2, "20210103223630Lh": 1, "20210201020330Lh": 3, "20210222222850Bh": 2, "20210223222830Ch": 2, "20210329015250Uh": 2, "20210329114430Ch": 0, "20210331231510Mh": 1, "20210616142650Bh": 0, "20210620163530Mh": 0, "20210621133650Bh": 3, "20210624133950Bh": 2, "20210626060050Uh": 1, "20210626133950Bh": 0, "20210721123430Ch": 3, "20210728164530Mh": 3, "20210731074710Th": 1, "20210810121630Ch": 2, "20210812044930Lh": 0, "20210814044930Lh": 3, "20210815140550Bh": 3, "20210817165030Mh": 3, "20211016145050Bh": 2, "20211119103130Ch": 1, "20211222120130Ch": 2, "20211225025850Uh": 0, "20211229120130Ch": 2, "20211229120210Th": 2, "20211231223430Lh": 2, "20220104043050Uh": 3, "20220122085210Th": 1, "20220328143550Bh": 0, "20220418020950Uh": 3, "20220617185352Mh": 0, "20220618185332Bh": 1, "20220622205632Bh": 2, "20220710085152Th": 3, "20220718185352Mh": 3, "20220721185332Bh": 3, "20220724185352Mh": 0, "20220803074752Th": 0}
HOLD = 0
MODEL, IMGSZ, EPOCHS, BATCH = "yolo11m-seg.pt", 1024, 50, 6
OUT = "/kaggle/working"; DS = "/kaggle/temp/yolo"
T0 = time.time()
def log(*a): print(f"[{(time.time()-T0)/60:6.1f}m]", *a, flush=True)

js = glob.glob("/kaggle/input/**/MAGFiLO_1.0_Annotations_kaggle2026_train.json", recursive=True)[0]
root = os.path.dirname(os.path.dirname(js))
tr_dir, te_dir = f"{root}/train/train_images", f"{root}/test/test_images"
coco = json.load(open(js))
anns = {}
for a in coco["annotations"]:
    anns.setdefault(a["image_id"], []).append(a)
for split in ("train", "val"):
    os.makedirs(f"{DS}/images/{split}", exist_ok=True); os.makedirs(f"{DS}/labels/{split}", exist_ok=True)
n = {"train": 0, "val": 0}
for im in coco["images"]:
    stem = os.path.splitext(im["file_name"])[0]
    split = "val" if FOLD[stem] == HOLD else "train"
    name = im["id"].replace("/", "_")                    # one sample per annotator set
    os.symlink(f"{tr_dir}/{im['file_name']}", f"{DS}/images/{split}/{name}.jpeg")
    with open(f"{DS}/labels/{split}/{name}.txt", "w") as f:
        for a in anns.get(im["id"], []):
            p = np.array(max(a["segmentation"], key=len), float) / 2048.0
            if len(p) >= 6:
                f.write("0 " + " ".join(f"{x:.5f}" for x in np.clip(p, 0, 1)) + "\n")
    n[split] += 1
open(f"{DS}/data.yaml", "w").write(f"path: {DS}\ntrain: images/train\nval: images/val\nnames:\n  0: filament\n")
log("samples", n)

model = YOLO(MODEL)
model.train(data=f"{DS}/data.yaml", imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, device=0, workers=4, seed=0,
            cos_lr=True, close_mosaic=10, degrees=180, flipud=0.5, fliplr=0.5, hsv_h=0, hsv_s=0, mask_ratio=2,
            overlap_mask=False, max_det=150, project=f"{OUT}/runs", name="seg", exist_ok=True, plots=False, verbose=False)
w = f"{OUT}/runs/seg/weights/last.pt"                    # last, not best: best is picked on the holdout
log("trained", w)

def predict(paths):
    m = YOLO(w); res = {}
    for i, p in enumerate(paths):
        r = m.predict(p, imgsz=IMGSZ, conf=0.03, iou=0.6, max_det=150, retina_masks=True, verbose=False)[0]
        out = []
        if r.masks is not None:
            ms = r.masks.data.cpu().numpy() > 0.5
            for s, mk in zip(r.boxes.conf.cpu().numpy(), ms):
                if mk.shape != (2048, 2048):
                    mk = cv2.resize(mk.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST) > 0
                if mk.sum() >= 20:
                    out.append((float(s), mu.encode(np.asfortranarray(mk.astype(np.uint8)))["counts"].decode()))
        res[os.path.splitext(os.path.basename(p))[0]] = out
        if i % 50 == 0: log("pred", i, len(paths))
    return res

hold = sorted(f"{tr_dir}/{s}.jpeg" for s, f in FOLD.items() if f == HOLD)
pickle.dump(predict(hold), open(f"{OUT}/pred_hold.pkl", "wb"))
pickle.dump(predict(sorted(glob.glob(f"{te_dir}/*.jpeg"))), open(f"{OUT}/pred_test.pkl", "wb"))
log("done")


## References and final evaluation materials

- Competition and final submission instructions: https://www.kaggle.com/competitions/filament-segmentation-2026/overview/announcements
- Official PQ self-evaluator: https://www.kaggle.com/code/azimahmadzadeh/self-evaluation-notebook
- Kirillov et al., Panoptic Segmentation, CVPR 2019: https://doi.org/10.1109/CVPR.2019.00963
- Segmentation Models PyTorch: https://github.com/qubvel-org/segmentation_models.pytorch
- Ultralytics YOLO: https://github.com/ultralytics/ultralytics

The organizers additionally require a public Git repository, pinned requirements, a four-page report in their
specified template, and a final Google Form submission by November 15, 2026. A Kaggle CSV alone does not
complete that final evaluation application. See the source release's status file for outstanding items.